# 02 · OpenFF vs external water models (theme B)

The sharpest test of the virtual-site export: OpenFF's four-/five-site water
force fields are exported by Interchange as OpenMM **`LocalCoordinatesSite`**
particles, whereas the canonical OpenMM/AMBER water models
(`tip5p.xml`, `amber/tip4pew_standard.xml`, `amber/opc_standard.xml`) use
**`OutOfPlaneSite`** / **`ThreeParticleAverageSite`**. Given identical real-atom
coordinates, the two encodings must place the M-site at the same point and yield
the same nonbonded parameters and energies.

We compare, for **TIP5P, TIP4P-Ew, and OPC**:
1. virtual-site placement (headline: exact to machine precision for the rigid
geometry the model actually simulates),
2. per-particle charges / Lennard-Jones parameters and nonbonded exclusions,
3. total energy of a periodic water box under PME.

We also document a subtlety: for an *arbitrary, non-ideal* water geometry the
`LocalCoordinatesSite` and `OutOfPlaneSite` frames diverge slightly — expected,
and irrelevant in practice because rigid-water constraints hold the geometry at
the model's ideal values (and vsites are recomputed from the constrained
coordinates).

In [1]:
import warnings

import numpy as np
import openmm
import openmm.unit as u
from openff.toolkit import ForceField as OFF
from openff.toolkit import Molecule
from openff.toolkit import Topology as OFFTopology
from openmm.app import Element, Modeller, NoCutoff, PME
from openmm.app import ForceField as OMMFF
from openmm.app import Topology as OMMTopology

import _helpers as H

warnings.filterwarnings("ignore")
np.set_printoptions(precision=6, suppress=True)
RESULTS = []

MODELS = {
    "TIP5P":    dict(offxml="tip5p.offxml",    xml="tip5p.xml"),
    "TIP4P-Ew": dict(offxml="tip4p_ew.offxml", xml="amber/tip4pew_standard.xml"),
    "OPC":      dict(offxml="opc.offxml",      xml="amber/opc_standard.xml"),
}

## Builders

`openff_water_system` builds the Interchange (`LocalCoordinatesSite`) system;
`openmm_water_system` builds the OpenMM built-in system, adding M-sites with
`Modeller.addExtraParticles`. Both start from the *same* O/H/H coordinates.

In [2]:
def make_water():
    w = Molecule.from_mapped_smiles("[O:1]([H:2])[H:3]")
    w.generate_conformers(n_conformers=1)
    return w


def openff_water_system(offxml, waters):
    off_top = OFFTopology.from_molecules(waters)
    return OFF(offxml).create_openmm_system(off_top)


def openmm_water_topology(n):
    top = OMMTopology()
    chain = top.addChain()
    for _ in range(n):
        r = top.addResidue("HOH", chain)
        O = top.addAtom("O", Element.getBySymbol("O"), r)
        H1 = top.addAtom("H1", Element.getBySymbol("H"), r)
        H2 = top.addAtom("H2", Element.getBySymbol("H"), r)
        top.addBond(O, H1)
        top.addBond(O, H2)
    return top


def openmm_water_system(xml, oh_h_positions, n, *, nonbonded_method=NoCutoff,
                        box=None, **create_kwargs):
    ff = OMMFF(xml)
    top = openmm_water_topology(n)
    if box is not None:
        top.setPeriodicBoxVectors(box)
    modeller = Modeller(top, oh_h_positions)
    modeller.addExtraParticles(ff)
    system = ff.createSystem(modeller.topology, nonbondedMethod=nonbonded_method,
                             rigidWater=True, **create_kwargs)
    return system, modeller.positions


def vsite_positions(system, positions, apply_constraints):
    full = H.resolved_vsite_positions(system, positions, apply_constraints=apply_constraints)
    idx = [i for i in range(system.getNumParticles()) if system.isVirtualSite(i)]
    return full[idx]


def max_vsite_gap(vs_a, vs_b):
    return max(min(np.linalg.norm(a - b) for b in vs_b) for a in vs_a)

## 1 & 2 · Single-molecule vsite placement, charges, LJ, exclusions

Headline check uses the constrained (ideal, rigidly-simulated) geometry.

In [3]:
POS_TOL_A = 1e-6      # Angstrom, machine-precision agreement for ideal geometry
PARAM_TOL = 1e-6


def compare_single(model):
    offxml, xml = MODELS[model]["offxml"], MODELS[model]["xml"]
    w = make_water()
    coords = w.conformers[0].to_openmm()

    ic = openff_water_system(offxml, [w])
    og, omm_pos = openmm_water_system(xml, coords, 1)

    n_vs = H.count_vsites(ic)
    assert n_vs == H.count_vsites(og) > 0
    off_types = {ic.getVirtualSite(i).__class__.__name__
                 for i in range(ic.getNumParticles()) if ic.isVirtualSite(i)}
    omm_types = {og.getVirtualSite(i).__class__.__name__
                 for i in range(og.getNumParticles()) if og.isVirtualSite(i)}

    # vsite placement: constrained (ideal) vs unconstrained (informational)
    off_vs_c = vsite_positions(ic, coords, apply_constraints=True)
    omm_vs_c = vsite_positions(og, omm_pos, apply_constraints=True)
    gap_c = max_vsite_gap(off_vs_c, omm_vs_c) * 10.0
    off_vs_u = vsite_positions(ic, coords, apply_constraints=False)
    omm_vs_u = vsite_positions(og, omm_pos, apply_constraints=False)
    gap_u = max_vsite_gap(off_vs_u, omm_vs_u) * 10.0

    print(f"\n=== {model} ===")
    print(f"  vsites: {n_vs}   OpenFF type={off_types}   OpenMM type={omm_types}")
    print(f"  vsite placement gap (ideal/constrained)  : {gap_c:.3e} A")
    print(f"  vsite placement gap (raw/unconstrained)  : {gap_u:.3e} A  (expected small, frame-dependent)")
    assert gap_c <= POS_TOL_A, f"{model}: constrained vsite placement gap {gap_c} A too large"

    # charges / LJ. Compare physically: charge and epsilon must match exactly;
    # sigma matters only where epsilon > 0 (an LJ-less vsite's sigma is an inert
    # placeholder). We therefore compare the (charge, epsilon) multiset and the
    # sigma multiset restricted to epsilon>0 particles.
    def rows(system):
        out = []
        for i in range(system.getNumParticles()):
            p = H.particle_nb_params(system, i)
            out.append((round(p["charge"], 6), round(p["sigma"], 5), round(p["epsilon"], 6)))
        return out
    rf, rm = rows(ic), rows(og)
    ce_f = sorted((c, e) for c, s, e in rf)
    ce_m = sorted((c, e) for c, s, e in rm)
    charge_ok = all(abs(a[0] - b[0]) <= PARAM_TOL for a, b in zip(ce_f, ce_m))
    eps_ok = all(abs(a[1] - b[1]) <= PARAM_TOL for a, b in zip(ce_f, ce_m))
    sig_f = sorted(s for c, s, e in rf if e > 0)
    sig_m = sorted(s for c, s, e in rm if e > 0)
    lj_ok = (len(sig_f) == len(sig_m)
             and all(abs(a - b) <= 1e-4 for a, b in zip(sig_f, sig_m)))
    # placeholder-sigma difference on LJ-less (epsilon==0) particles is inert
    inert_f = sorted(s for c, s, e in rf if e == 0)
    inert_m = sorted(s for c, s, e in rm if e == 0)
    inert_sigma_note = "identical" if inert_f == inert_m else \
        f"differ (OpenFF {inert_f} vs OpenMM {inert_m}) — inert, epsilon=0"
    print(f"  charge multiset match: {charge_ok}   epsilon match: {eps_ok}   "
          f"LJ-active sigma match: {lj_ok}")
    print(f"  LJ-less placeholder sigma: {inert_sigma_note}")
    assert charge_ok, f"{model}: charge mismatch\n OFF {rf}\n OMM {rm}"
    assert eps_ok, f"{model}: epsilon mismatch\n OFF {rf}\n OMM {rm}"
    assert lj_ok, f"{model}: LJ-active sigma mismatch\n OFF {rf}\n OMM {rm}"

    # exclusions: both should fully exclude all intra-water pairs
    n_exc_ic = len(H.exceptions_table(ic))
    n_exc_og = len(H.exceptions_table(og))
    print(f"  intra-water exceptions: OpenFF={n_exc_ic}  OpenMM={n_exc_og}")
    assert n_exc_ic == n_exc_og, f"{model}: exception count differs"

    RESULTS.append({"notebook": "02", "model": model, "n_vsites": n_vs,
                    "off_vsite_type": sorted(off_types), "omm_vsite_type": sorted(omm_types),
                    "vsite_gap_constrained_A": gap_c, "vsite_gap_raw_A": gap_u,
                    "charge_match": bool(charge_ok), "lj_match": bool(lj_ok),
                    "status": "PASS"})


for model in MODELS:
    compare_single(model)
print("\nSingle-molecule placement/parameter equivalence: PASS for all models")


=== TIP5P ===
  vsites: 2   OpenFF type={'LocalCoordinatesSite'}   OpenMM type={'OutOfPlaneSite'}
  vsite placement gap (ideal/constrained)  : 2.893e-08 A
  vsite placement gap (raw/unconstrained)  : 2.538e-02 A  (expected small, frame-dependent)
  charge multiset match: True   epsilon match: True   LJ-active sigma match: True
  LJ-less placeholder sigma: identical
  intra-water exceptions: OpenFF=10  OpenMM=10

=== TIP4P-Ew ===
  vsites: 1   OpenFF type={'LocalCoordinatesSite'}   OpenMM type={'ThreeParticleAverageSite'}
  vsite placement gap (ideal/constrained)  : 8.730e-09 A
  vsite placement gap (raw/unconstrained)  : 7.748e-03 A  (expected small, frame-dependent)
  charge multiset match: True   epsilon match: True   LJ-active sigma match: True
  LJ-less placeholder sigma: differ (OpenFF [1.0, 10.0, 10.0] vs OpenMM [10.0, 10.0, 10.0]) — inert, epsilon=0
  intra-water exceptions: OpenFF=6  OpenMM=6

=== OPC ===
  vsites: 1   OpenFF type={'LocalCoordinatesSite'}   OpenMM type={'Three

## Adversarial — many random rigid orientations

The `LocalCoordinatesSite` encoding must be rotation/translation equivariant. We
take each model's ideal (constrained) water, apply many random rigid-body
transforms, and require the OpenFF and OpenMM M-sites to coincide every time.

In [4]:
def random_rotation(rng):
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)],
    ])


def adversarial_orientations(model, n_trials=50, seed=1):
    offxml, xml = MODELS[model]["offxml"], MODELS[model]["xml"]
    w = make_water()
    ic = openff_water_system(offxml, [w])
    # ideal O,H,H from the constrained OpenFF system (first 3 particles)
    base = H.resolved_vsite_positions(ic, w.conformers[0].to_openmm(),
                                      apply_constraints=True)[:3]
    rng = np.random.default_rng(seed)
    worst = 0.0
    for _ in range(n_trials):
        R = random_rotation(rng)
        t = rng.uniform(-1, 1, size=3)
        oh = (base @ R.T) + t
        pos = oh * u.nanometer
        og, omm_pos = openmm_water_system(xml, pos, 1)
        off_vs = vsite_positions(ic, pos, apply_constraints=True)
        omm_vs = vsite_positions(og, omm_pos, apply_constraints=True)
        worst = max(worst, max_vsite_gap(off_vs, omm_vs) * 10.0)
    print(f"  {model}: worst vsite gap over {n_trials} random poses = {worst:.3e} A")
    assert worst <= 1e-5, f"{model}: orientation-dependent vsite gap {worst} A"
    RESULTS.append({"notebook": "02", "model": f"{model} (orientations)",
                    "vsite_gap_constrained_A": worst, "status": "PASS"})


print("Random rigid-orientation equivariance:")
for model in MODELS:
    adversarial_orientations(model)

Random rigid-orientation equivariance:
  TIP5P: worst vsite gap over 50 random poses = 2.893e-08 A


  TIP4P-Ew: worst vsite gap over 50 random poses = 8.730e-09 A


  OPC: worst vsite gap over 50 random poses = 1.863e-08 A


## 3 · Periodic water-box energy under PME

We build a cubic lattice of ideal, randomly-oriented waters and parameterize the
*identical* coordinates + box both ways, both with PME at a common Ewald
tolerance. Total energies must agree closely (residual is PME-grid noise).

In [5]:
def water_box(model, n_side=4, spacing=0.31, seed=7):
    """Return (off_top_positions, box, waters, coords_quantity) on a lattice."""
    w = make_water()
    ic = openff_water_system(model_offxml := MODELS[model]["offxml"], [w])
    ideal = H.resolved_vsite_positions(ic, w.conformers[0].to_openmm(),
                                       apply_constraints=True)[:3]
    ideal = ideal - ideal[0]  # O at origin
    rng = np.random.default_rng(seed)
    coords = []
    for i in range(n_side):
        for j in range(n_side):
            for k in range(n_side):
                R = random_rotation(rng)
                center = np.array([i, j, k]) * spacing
                coords.append((ideal @ R.T) + center)
    coords = np.vstack(coords)
    n = n_side ** 3
    L = n_side * spacing
    box = np.eye(3) * L
    return coords, box, [make_water() for _ in range(n)], n


def compare_box(model):
    coords, box, waters, n = water_box(model)
    boxq = box * u.nanometer
    cutoff = min(0.9, (box[0, 0] / 2) - 0.01)

    # OpenFF path
    off_top = OFFTopology.from_molecules(waters)
    off_top.box_vectors = boxq
    ic = OFF(MODELS[model]["offxml"]).create_interchange(off_top).to_openmm_system(
        combine_nonbonded_forces=True)
    # OpenMM path (same coords)
    og, omm_pos = openmm_water_system(MODELS[model]["xml"], coords * u.nanometer, n,
                                      nonbonded_method=PME, box=boxq)
    for s in (ic, og):
        for f in s.getForces():
            if isinstance(f, openmm.NonbondedForce):
                f.setEwaldErrorTolerance(1e-6)
                f.setUseSwitchingFunction(False)   # match layout; avoid switch-distance mismatch
                f.setCutoffDistance(cutoff * u.nanometer)
        s.setDefaultPeriodicBoxVectors(*boxq)

    # Both systems' real atoms are O,H,H per water in the same water order, so we
    # feed each system the same real-atom positions and let it recompute its own
    # vsites (their parent-atom orderings differ between the two encodings, which
    # is irrelevant once vsites are recomputed from parents).
    real_pos = H.real_atom_positions(og, omm_pos)
    e_og, _ = H.compute_energy(og, omm_pos, recompute_vsites=True, platform="CPU")
    e_ic, _ = H.compute_energy(ic, real_pos, recompute_vsites=True, platform="CPU")
    eo = e_og["total"].value_in_unit(H.ENERGY_UNIT)
    ei = e_ic["total"].value_in_unit(H.ENERGY_UNIT)
    rel = abs(eo - ei) / max(abs(eo), 1.0)
    print(f"  {model}: box {n} waters  OpenMM={eo:.4f}  OpenFF={ei:.4f} kcal/mol  "
          f"rel diff={rel:.2e}")
    assert rel <= 1e-4, f"{model}: box energy relative difference {rel:.2e} too large"
    RESULTS.append({"notebook": "02", "model": f"{model} (PME box)", "n_waters": n,
                    "energy_openmm_kcal": eo, "energy_openff_kcal": ei,
                    "rel_diff": rel, "status": "PASS"})


print("\nPeriodic PME water-box energy consistency:")
for model in MODELS:
    compare_box(model)


Periodic PME water-box energy consistency:


  TIP5P: box 64 waters  OpenMM=-90.7446  OpenFF=-90.7446 kcal/mol  rel diff=2.76e-08


  TIP4P-Ew: box 64 waters  OpenMM=-74.9584  OpenFF=-74.9584 kcal/mol  rel diff=2.07e-07


  OPC: box 64 waters  OpenMM=-90.8252  OpenFF=-90.8252 kcal/mol  rel diff=1.44e-07


In [6]:
import json
with open("results_02.json", "w") as fh:
    json.dump(RESULTS, fh, indent=2, default=str)
print(f"\nWrote results_02.json ({len(RESULTS)} rows)")
print("Notebook 02 complete: OpenFF LocalCoordinatesSite water models reproduce the "
      "external OutOfPlane/ThreeParticleAverage references (placement, parameters, PME energy).")


Wrote results_02.json (9 rows)
Notebook 02 complete: OpenFF LocalCoordinatesSite water models reproduce the external OutOfPlane/ThreeParticleAverage references (placement, parameters, PME energy).
